# Clarté Commerce — Cohort Retention Analysis

**Client:** Clarté Commerce S.A.S.  
**Analyst:** Nurbol Sultanov  
**Date:** March 2026  

Track customer retention by monthly acquisition cohort.  
Compare retention curves before and after the Q3 2023 rebranding.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

sys.path.append('../src')
from data_cleaning import load_transactions, load_customers, remove_test_orders

REBRAND_DATE = pd.Timestamp('2023-08-15')

In [2]:
transactions = load_transactions()
customers = load_customers()
transactions, _ = remove_test_orders(transactions)

print(f'Clean transactions: {len(transactions):,}')

Clean transactions: 92,310


## 1. Build Cohort Matrix

In [3]:
# Assign each customer to their first-purchase cohort
transactions['order_month'] = transactions['transaction_date'].dt.to_period('M')

cohort_month = transactions.groupby('customer_id')['order_month'].min().rename('cohort_month')
transactions = transactions.merge(cohort_month, on='customer_id')

# Calculate month offset
transactions['month_offset'] = (transactions['order_month'] - transactions['cohort_month']).apply(lambda x: x.n)

# Build cohort matrix
cohort_data = transactions.groupby(['cohort_month', 'month_offset'])['customer_id'].nunique().reset_index()
cohort_data.columns = ['cohort_month', 'month_offset', 'active_customers']

cohort_pivot = cohort_data.pivot(index='cohort_month', columns='month_offset', values='active_customers')

# Retention percentages
cohort_sizes = cohort_pivot[0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

print(f'Cohort matrix shape: {retention_matrix.shape}')
print(f'Cohorts from {retention_matrix.index.min()} to {retention_matrix.index.max()}')

Cohort matrix shape: (36, 36)
Cohorts from 2022-01 to 2024-12


## 2. Retention Heatmap

In [4]:
# Trim to first 12 months for readability
retention_12m = retention_matrix.iloc[:, :13]

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    retention_12m,
    annot=True, fmt='.0f', 
    cmap='RdYlGn',
    vmin=0, vmax=100,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Monthly Cohort Retention (%) \u2014 First 12 Months', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
ax.set_yticklabels([str(x) for x in retention_12m.index], rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/cohort_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Retention Curves: Pre vs Post Rebrand

In [5]:
# Split cohorts into pre and post rebrand
rebrand_period = pd.Period('2023-08', freq='M')

pre_cohorts = retention_matrix[retention_matrix.index < rebrand_period]
post_cohorts = retention_matrix[retention_matrix.index >= rebrand_period]

# Average retention curve for each group
pre_avg = pre_cohorts.iloc[:, :13].mean()
post_avg = post_cohorts.iloc[:, :13].mean()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(pre_avg.index, pre_avg.values, 'o-', color='#3498db', linewidth=2, markersize=6, label='Pre-Rebrand Cohorts')
ax.plot(post_avg.index, post_avg.values, 's-', color='#e74c3c', linewidth=2, markersize=6, label='Post-Rebrand Cohorts')

ax.set_title('Average Retention Curve \u2014 Pre vs Post Rebrand Cohorts', fontsize=13, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Retention (%)')
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/retention_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [6]:
print('Average retention at key milestones:\n')
print(f'{"":14s} {"Pre-Rebrand":>12s} {"Post-Rebrand":>13s} {"Diff":>8s}')
for m in [1, 3, 6, 12]:
    if m in pre_avg.index and m in post_avg.index:
        pre_val = pre_avg[m]
        post_val = post_avg[m]
        diff = post_val - pre_val
        print(f'Month {m:<8d} {pre_val:>10.1f}%  {post_val:>11.1f}%  {diff:>+7.1f}pp')

Average retention at key milestones:

              Pre-Rebrand  Post-Rebrand   Diff
Month 1          42.3%        35.8%      -6.5pp
Month 3          31.7%        24.2%      -7.5pp
Month 6          24.1%        17.6%      -6.5pp
Month 12         18.4%        12.1%      -6.3pp


## 4. Rebrand Impact on Existing Cohorts

Look at pre-rebrand cohorts and check if their retention drops after the rebrand date.

In [7]:
# For each pre-rebrand cohort, find the month offset when rebrand happened
# and compare retention before vs after that point

selected_cohorts = ['2022-01', '2022-07', '2023-01', '2023-07']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, cohort_str in zip(axes.flat, selected_cohorts):
    cohort_period = pd.Period(cohort_str, freq='M')
    if cohort_period in retention_matrix.index:
        curve = retention_matrix.loc[cohort_period].dropna()
        
        # Find month offset when rebrand happened
        rebrand_offset = (rebrand_period - cohort_period).n
        
        ax.plot(curve.index, curve.values, 'o-', color='#2c3e50', linewidth=1.5, markersize=4)
        if rebrand_offset > 0 and rebrand_offset < len(curve):
            ax.axvline(x=rebrand_offset, color='red', linestyle='--', alpha=0.7, label='Rebrand')
        ax.set_title(f'Cohort {cohort_str}')
        ax.set_xlabel('Month Offset')
        ax.set_ylabel('Retention (%)')
        ax.set_ylim(0, 105)
        ax.legend(fontsize=9)

plt.suptitle('Retention Curves for Selected Pre-Rebrand Cohorts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Export

In [8]:
retention_matrix.to_csv('../data/processed/cohort_matrix.csv')
print('Exported cohort_matrix.csv')

Exported cohort_matrix.csv


## Key Findings

1. **Post-rebrand cohorts retain ~6-7pp worse** at every milestone (M1, M3, M6, M12)
2. **Pre-rebrand cohorts show a retention cliff** right after the rebrand date \u2014 existing customers were disrupted
3. **M1 retention dropped from 42% to 36%** \u2014 first-month engagement is weaker post-rebrand
4. **The gap persists**: it's not just initial shock, retention stays lower through M12
5. **2023-07 cohort** (acquired just before rebrand) shows the sharpest drop \u2014 onboarded under old brand, experienced new brand immediately

**Implication:** The rebrand didn't just lose existing customers \u2014 it also reduced the stickiness of newly acquired ones.